## `torch.nn.Module` and `torch.nn.Parameter`

In [39]:
from typing import Any

import torch


class TinyModel(torch.nn.Module):

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)

        self.linear1 = torch.nn.Linear(100, 200)
        self.activation = torch.nn.ReLU()
        self.linear2 = torch.nn.Linear(200, 10)
        self.softmax = torch.nn.Softmax(dim=1)

    def forward(self, x):
        x = self.linear1(x)
        x = self.activation(x)
        x = self.linear2(x)
        x = self.softmax(x)
        return x


tinymodel = TinyModel()

print("The model:")
print(tinymodel)

print("\nJust one layer:")
print(tinymodel.linear2)

print("\n\nModel params:")
for param in tinymodel.parameters():
    print(param)

print("\n\nLayer params:")
for param in tinymodel.linear2.parameters():
    print(param)

The model:
TinyModel(
  (linear1): Linear(in_features=100, out_features=200, bias=True)
  (activation): ReLU()
  (linear2): Linear(in_features=200, out_features=10, bias=True)
  (softmax): Softmax(dim=1)
)

Just one layer:
Linear(in_features=200, out_features=10, bias=True)


Model params:
Parameter containing:
tensor([[ 0.0523, -0.0628,  0.0775,  ..., -0.0291, -0.0184, -0.0860],
        [ 0.0098, -0.0678, -0.0425,  ..., -0.0174,  0.0367,  0.0016],
        [ 0.0400,  0.0647,  0.0085,  ...,  0.0133,  0.0554,  0.0517],
        ...,
        [-0.0554, -0.0439,  0.0131,  ..., -0.0126, -0.0395,  0.0009],
        [ 0.0981, -0.0477,  0.0024,  ...,  0.0519,  0.0912, -0.0363],
        [ 0.0741, -0.0122, -0.0694,  ..., -0.0679, -0.0090, -0.0013]],
       requires_grad=True)
Parameter containing:
tensor([-0.0053,  0.0352, -0.0191, -0.0833,  0.0365,  0.0846,  0.0114, -0.0966,
         0.0150,  0.0371,  0.0890, -0.0299,  0.0712, -0.0013,  0.0112, -0.0499,
         0.0242,  0.0344,  0.0762,  0.0283, 

## Common Layer Types

### Linear Layers

In [46]:
lin = torch.nn.Linear(3, 2)
x = torch.rand(1, 3)
print("Input:")
print(x)

print("\n\nWeight and Bias parameters:")
for param in lin.parameters():
    print(param)

print(f"{lin.weight=}")
print(f"{lin.bias=}")

y = lin(x)
print("\n\nOutput:")
print(y)

Input:
tensor([[0.8955, 0.2789, 0.1983]])


Weight and Bias parameters:
Parameter containing:
tensor([[-0.1198, -0.2892, -0.5463],
        [-0.5563, -0.2272, -0.4421]], requires_grad=True)
Parameter containing:
tensor([-0.4019,  0.3975], requires_grad=True)
lin.weight=Parameter containing:
tensor([[-0.1198, -0.2892, -0.5463],
        [-0.5563, -0.2272, -0.4421]], requires_grad=True)
lin.bias=Parameter containing:
tensor([-0.4019,  0.3975], requires_grad=True)


Output:
tensor([[-0.6982, -0.2517]], grad_fn=<AddmmBackward0>)


### Convolutional Layers

In [48]:
from typing import Any

import torch.nn.functional as F


class LeNet(torch.nn.Module):

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)

        # 1 input image channel (black & white), 6 output channels, 5x5 square convolution kernel
        self.conv1 = torch.nn.Conv2d(1, 6, 5)
        self.conv2 = torch.nn.Conv2d(6, 16, 5)

        # an affine operation: y = Wx + b
        self.fc1 = torch.nn.Linear(16 * 6 * 6, 120)
        self.fc2 = torch.nn.Linear(120, 84)
        self.fc3 = torch.nn.Linear(84, 10)

    def forward(self, x):
        # Max pooling over a (2, 2) window
        x = F.max_pool2d(F.relu(self.conv1(x)), (2, 2))
        # If the size is a square you can only specify a single number
        x = F.max_pool2d(F.relu(self.conv2(x)), (2, 2))
        x = x.view(-1, self.num_flat_features(x))
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

    def num_flat_features(self, x):
        size = x.size()[1:]  # all dimensions except the batch dimension
        num_features = 1
        for s in size:
            num_features *= s
        return num_features

### Recurrent Layers

In [ ]:
from typing import Any


class LSTMTagger(torch.nn.Module):

    def __init__(self, embedding_dim, hidden_dim, vocab_size, target_size) -> None:
        super().__init__()

        self.hidden_dim = hidden_dim

        self.word_embeddings = torch.nn.Embedding(vocab_size, embedding_dim)

        # The LSTM takes word embeddings as inputs, and outputs hidden states
        # with dimensionality hidden_dim
        self.lstm = torch.nn.LSTM(embedding_dim, hidden_dim, batch_first=True)

        # The linear layer that maps from hidden state space th tag space
        self.hidden2tag = torch.nn.Linear(hidden_dim, target_size)

    def forward(self, sentence):
        # sentence 现在期望形状为 (batch_size, seq_len)
        batch_size, seq_len = sentence.size()  # 获取两个维度
        embeds = self.word_embeddings(sentence)  # (batch, seq_len, embedding_dim)
        lstm_out, _ = self.lstm(embeds)  # (batch, seq_len, hidden_dim)
        tag_space = self.hidden2tag(lstm_out)  # (batch, seq_len, target_size)
        # 改动2：log_softmax 作用在最后一维（类别维）
        tag_scores = F.log_softmax(tag_space, dim=2)
        return tag_scores

In [67]:
from torchinfo import summary

lstmtagger = LSTMTagger(16, 32, 128, 10)
summary(
    lstmtagger,
    input_size=(4, 8),
    dtypes=[torch.int],
    col_names=["input_size", "output_size", "num_params"],
)

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #
LSTMTagger                               [4, 8]                    [4, 8, 10]                --
├─Embedding: 1-1                         [4, 8]                    [4, 8, 16]                2,048
├─LSTM: 1-2                              [4, 8, 16]                [4, 8, 32]                6,400
├─Linear: 1-3                            [4, 8, 32]                [4, 8, 10]                330
Total params: 8,778
Trainable params: 8,778
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.21
Input size (MB): 0.00
Forward/backward pass size (MB): 0.01
Params size (MB): 0.04
Estimated Total Size (MB): 0.05

### Transformers

## Other Layers and Functions

In [ ]:
my_tensor = torch.rand(1, 6, 6)
print(my_tensor)

maxpool_layer = torch.nn.MaxPool2d(3)
print(maxpool_layer(my_tensor))

tensor([[[0.4254, 0.4614, 0.3564, 0.8980, 0.7153, 0.0767],
         [0.6555, 0.9025, 0.7197, 0.7015, 0.0118, 0.2813],
         [0.6279, 0.0592, 0.4474, 0.2160, 0.2856, 0.2647],
         [0.7006, 0.1380, 0.9573, 0.2216, 0.0515, 0.5192],
         [0.0410, 0.5917, 0.8434, 0.0715, 0.6100, 0.8634],
         [0.7649, 0.5492, 0.2098, 0.4755, 0.6165, 0.0830]]])
tensor([[[0.9025, 0.8980],
         [0.9573, 0.8634]]])


In [76]:
my_tensor = torch.rand(1, 4, 4) * 20 + 5
print(my_tensor)

print(f"{my_tensor.mean()=}")
print(f"{my_tensor.std()=}")
print()

norm_layer = torch.nn.BatchNorm1d(4)
normed_tensor = norm_layer(my_tensor)
print(normed_tensor)

print(f"{normed_tensor.mean()=}")
print(f"{normed_tensor.std()=}")

tensor([[[ 5.5859, 24.0913,  8.5065, 12.0961],
         [10.3701, 20.9270, 16.9857, 12.5932],
         [14.2653, 18.2085,  7.8786, 14.9321],
         [16.6274, 10.8705, 19.9447, 15.5173]]])
my_tensor.mean()=tensor(14.3375)
my_tensor.std()=tensor(5.0802)

tensor([[[-0.9920,  1.6365, -0.5772, -0.0673],
         [-1.1927,  1.4041,  0.4346, -0.6459],
         [ 0.1187,  1.1726, -1.5883,  0.2969],
         [ 0.2731, -1.4987,  1.2941, -0.0685]]],
       grad_fn=<NativeBatchNormBackward0>)
normed_tensor.mean()=tensor(-1.4901e-08, grad_fn=<MeanBackward0>)
normed_tensor.std()=tensor(1.0328, grad_fn=<StdBackward0>)


In [78]:
my_tensor = torch.rand(1, 4, 4)
print(my_tensor)

dropout = torch.nn.Dropout(p=0.4)
print(dropout(my_tensor))
print(dropout(my_tensor))

tensor([[[0.7437, 0.7382, 0.1694, 0.3409],
         [0.3639, 0.3546, 0.7136, 0.5015],
         [0.7398, 0.6113, 0.1263, 0.2711],
         [0.5450, 0.5558, 0.6808, 0.6094]]])
tensor([[[1.2396, 0.0000, 0.0000, 0.5681],
         [0.0000, 0.0000, 1.1893, 0.0000],
         [1.2329, 1.0188, 0.0000, 0.0000],
         [0.9084, 0.9264, 0.0000, 1.0157]]])
tensor([[[1.2396, 0.0000, 0.2823, 0.5681],
         [0.6064, 0.0000, 1.1893, 0.0000],
         [0.0000, 1.0188, 0.2104, 0.4518],
         [0.9084, 0.9264, 0.0000, 0.0000]]])


### Activation Functions

### Loss Functions